In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import math

In [2]:
df = pd.read_csv("../00_datasets/dataset_output/libras_minds/libras_minds_openpose.csv")

In [4]:
df = df.drop(["Unnamed: 0"], axis=1)

In [5]:
person_videos = 15
persons = 5
frames = 60
person = []
for i in range(persons):
    person.extend([i] * frames * person_videos)
person = person * len(df["category"].unique())

In [6]:
person = df["video_name"].str[-8:-6]

In [7]:
df["person"] = person

In [8]:
df

,category,video_name,frame,hand_0_0_x,hand_0_0_y,hand_0_0_z,hand_0_1_x,hand_0_1_y,hand_0_1_z,hand_0_2_x,...,pose_22_z,pose_23_x,pose_23_y,pose_23_z,pose_24_x,pose_24_y,pose_24_z,missing_hand,missing_face,person
0,Acontecer,01AcontecerSinalizador01-1.mp4,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,False,01
1,Acontecer,01AcontecerSinalizador01-1.mp4,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,False,01
2,Acontecer,01AcontecerSinalizador01-1.mp4,2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,False,01
3,Acontecer,01AcontecerSinalizador01-1.mp4,3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,01
4,Acontecer,01AcontecerSinalizador01-1.mp4,4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161434,Vontade,20VontadeSinalizador12-5.mp4,162,0.614689,0.910987,0.588928,0.598457,0.922928,0.627919,0.588381,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,False,12
161435,Vontade,20VontadeSinalizador12-5.mp4,163,0.614785,0.908597,0.584819,0.598921,0.922699,0.580145,0.588155,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,False,12
161436,Vontade,20VontadeSinalizador12-5.mp4,164,0.614690,0.911093,0.602180,0.598085,0.925853,0.639054,0.588122,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,False,12
161437,Vontade,20VontadeSinalizador12-5.mp4,165,0.614768,0.910723,0.575162,0.599248,0.924518,0.586799,0.588717,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,False,12


In [9]:
df["category"].unique()

array(['Acontecer', 'Aluno', 'Amarelo', 'America', 'Aproveitar', 'Bala',
       'Banco', 'Banheiro', 'Barulho', 'Cinco', 'Conhecer', 'Espelho',
       'Esquina', 'Filho', 'Maca', 'Medo', 'Ruim', 'Sapo', 'Vacina',
       'Vontade'], dtype=object)

In [10]:
landmarks_name = np.array([i for i in list(df.columns) if i not in ["Unnamed: 0", "Unnamed: 0.1", "Unnamed: 0.2", "category", "category_index", "video_name", "frame", "missing_hand", "missing_face", "person"]])
landmarks_name = landmarks_name.reshape((int(landmarks_name.shape[0]/3), 3))

In [11]:
# GPT Code
def rotate_landmarks(landmarks, rotation_angle):
    # Assuming landmarks are in (x, y, z) format
    
    # Calculate the centroid (center) of the landmarks
#     centroid = np.mean(landmarks, axis=0)
    centroid = 0.5

    # Translate landmarks to the origin (center)
    translated_landmarks = landmarks - centroid
    # Create a rotation matrix for the given angle (in radians)
    rotation_matrix = np.array([[np.cos(rotation_angle), -np.sin(rotation_angle), 0],
                                 [np.sin(rotation_angle), np.cos(rotation_angle), 0],
                                 [0, 0, 1]])
    
    # Apply the rotation to each landmark
#     print(rotation_matrix)
    rotated_landmarks = np.dot(translated_landmarks, rotation_matrix)
    
    return rotated_landmarks + centroid

In [12]:
def zoom_landmarks(landmarks, zoom_factor):
    # Scale the landmarks by the zoom factor
    zoomed_landmarks = landmarks * zoom_factor
    
    return zoomed_landmarks

In [13]:
def translate_landmarks(landmarks, translation_vector):
    # Translate the landmarks by the given vector
    translated_landmarks = landmarks + translation_vector
    
    return translated_landmarks

In [14]:
def apply_transformation(frame, rotation, zoom, translate_x, translate_y):
    for lx, ly, lz in landmarks_name:
        result = rotate_landmarks(frame[[lx, ly, lz]], math.radians(rotation))
        frame[[lx, ly, lz]] = result
        result = zoom_landmarks(frame[[lx, ly, lz]], zoom)
        frame[[lx, ly, lz]] = result
        result = translate_landmarks(frame[[lx, ly, lz]], [translate_x, translate_x, 0])
        frame[[lx, ly, lz]] = result
    return frame

In [15]:
np.random.normal(0, 0.5, 10)

array([-0.28865027, -0.03713633,  0.1770119 ,  0.16093344, -0.12663296,
       -0.21559848, -0.09797358, -0.23565666, -0.40216085,  0.21258636])

In [16]:
sample_size = 0.4
samples_creation = 4
rotation_sigma = 4
zoom_sigma = 0.05
translate_x_sigma = 0.01
translate_y_sigma = 0.0

In [17]:
df_augmented = pd.DataFrame(columns=df.columns)
for category in df["category"].unique():
    print("Processing category", category)
    df_category = df[df["category"] == category]
    videos = df_category["video_name"].unique()
    videos = np.random.choice(videos, int(len(videos) * sample_size))
    for i in range(samples_creation):
        for video in videos:
            df_video = df[df["video_name"] == video].copy()
            rotation = np.random.normal(0, rotation_sigma)
            zoom = np.random.normal(0, zoom_sigma) + 1
            translate_x = np.random.normal(0, translate_x_sigma)
            translate_y = np.random.normal(0, translate_y_sigma)
            df_video = apply_transformation(df_video, rotation, zoom, translate_x, translate_y)
#             for index, frame in df_video.iterrows():
#                 frame = apply_transformation(frame, rotation, zoom, translate_x, translate_y)
#                 df_video[index] = frame
            df_video["video_name"] = f"augmented{i}-" + df_video["video_name"]
            df_augmented = pd.concat([df_augmented, df_video])

Processing category Acontecer
Processing category Aluno
Processing category Amarelo
Processing category America
Processing category Aproveitar
Processing category Bala
Processing category Banco
Processing category Banheiro
Processing category Barulho
Processing category Cinco
Processing category Conhecer
Processing category Espelho
Processing category Esquina
Processing category Filho
Processing category Maca
Processing category Medo
Processing category Ruim
Processing category Sapo
Processing category Vacina
Processing category Vontade


In [18]:
df_augmented

,category,video_name,frame,hand_0_0_x,hand_0_0_y,hand_0_0_z,hand_0_1_x,hand_0_1_y,hand_0_1_z,hand_0_2_x,...,pose_22_z,pose_23_x,pose_23_y,pose_23_z,pose_24_x,pose_24_y,pose_24_z,missing_hand,missing_face,person
604,Acontecer,augmented0-01AcontecerSinalizador02-1.mp4,0,0.602822,0.988466,0.249326,0.590586,1.024414,0.075896,0.585656,...,0.0,0.009205,-0.009732,0.0,0.009205,-0.009732,0.0,False,False,02
605,Acontecer,augmented0-01AcontecerSinalizador02-1.mp4,1,0.604113,0.986668,0.318819,0.591311,1.024499,0.096373,0.585840,...,0.0,0.009205,-0.009732,0.0,0.009205,-0.009732,0.0,False,False,02
606,Acontecer,augmented0-01AcontecerSinalizador02-1.mp4,2,0.602879,0.989878,0.291561,0.591412,1.026965,0.064079,0.582652,...,0.0,0.009205,-0.009732,0.0,0.009205,-0.009732,0.0,False,False,02
607,Acontecer,augmented0-01AcontecerSinalizador02-1.mp4,3,0.602879,0.989889,0.292151,0.591412,1.026973,0.064691,0.582653,...,0.0,0.009205,-0.009732,0.0,0.009205,-0.009732,0.0,False,False,02
608,Acontecer,augmented0-01AcontecerSinalizador02-1.mp4,4,0.604532,0.989219,0.264656,0.593330,1.022805,0.077459,0.588633,...,0.0,0.009205,-0.009732,0.0,0.009205,-0.009732,0.0,False,False,02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159877,Vontade,augmented3-20VontadeSinalizador10-5.mp4,97,0.598158,1.174887,0.018912,0.590218,1.176942,0.025051,0.619110,...,0.0,0.016173,-0.021439,0.0,0.016173,-0.021439,0.0,False,False,10
159878,Vontade,augmented3-20VontadeSinalizador10-5.mp4,98,0.594164,1.151470,0.018910,0.595991,1.173450,0.024747,0.621342,...,0.0,0.016173,-0.021439,0.0,0.016173,-0.021439,0.0,False,False,10
159879,Vontade,augmented3-20VontadeSinalizador10-5.mp4,99,0.581372,1.184292,0.020666,0.588031,1.183346,0.025255,0.669837,...,0.0,0.016173,-0.021439,0.0,0.016173,-0.021439,0.0,False,False,10
159880,Vontade,augmented3-20VontadeSinalizador10-5.mp4,100,0.601415,1.176209,0.019086,0.593503,1.177100,0.025086,0.619243,...,0.0,0.016173,-0.021439,0.0,0.016173,-0.021439,0.0,False,False,10


In [19]:
pd.concat([df, df_augmented]).to_csv("../00_datasets/dataset_output/libras_minds/libras_minds_openpose_augmented.csv")

In [126]:
# df_augmented.to_csv("datasets/lsa64_augmentation_test.csv")